# 🫀 퀘스트 46 · Q7-P0 — **SVDB P-위치 표** (Q7-S′ 전제 조건)

| | **MedKOS / `notebooks/quest46_q7p0_svdb_pdelin.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-U**(공개 delineator · `dwt\|raw` Se 0.9109) · **Q7-T**(BUT PDB 검증) |
| 규약 | **R16 · R22 · R27 ③ · R29 ②** |
| 학습 | **0회** — 신호처리만 · GPU 불필요 · **약 35분**(1회 캐시) |

## 왜 별도 런인가

Q7-U 가 자를 세웠다 — NeuroKit2 웨이블릿 구획이 BUT PDB 전문가 주석 대비
**Se 0.9109 / PPV 0.7145**. Q7-S′ 는 그 자로 뽑은 **P 정렬 특징**을 쓴다.

그런데 그 자는 **연속 신호**를 먹는데, `svdb_data5.npz` 는 **300샘플 비트 조각**만
갖고 있다. 조각을 되붙이면 되지 않나? **Q7-U 가 그걸 실측으로 기각했다** —
Voronoi 재구성만으로 F1 이 **0.7705 → 0.6132** 로 떨어진다(이음매가 파형이 된다).

그러니 **원 SVDB 연속 신호**에서 구획해야 한다. 그리고 그 결과를 비트 좌표로
되돌려야 하는데 — **npz 에 R 위치가 없다.**

## ★★ 이 런의 절반은 좌표 정합의 **구성적 증명**이다

`svdb_data5.npz` 는 `beat · y5 · y3 · pid · sym · pre_rr · post_rr · rhythm · rr_edge`
를 갖는다. R 표본 위치는 **없다**. 그래서 `mit-bih/svdb_labels.py` 의 비트 절단
로직을 **그대로 재현**해 R 위치를 되살린다.

```
원 SVDB(128Hz) → resample_poly(→360Hz) → BEAT_SYMS 마스크
              → R 재정합(±50ms 창 벡터크기 argmax) → AAMI5 필터 → 경계 검사
```

재현이 맞았다면 **`(pid, sym)` 수열이 npz 와 원소 단위로 같아야 한다.**
그게 「비트 i ↔ 내 R 위치 i」의 증명이고, **어긋나면 즉시 중단**한다.

> 이 퀘스트는 좌표 어긋남으로 두 번 데었다 — Q7-T 는 주석 확장자와 레코드 이름을
> **추측**해서 두 번 멈췄다. 이번엔 추측하지 않고 **증명**한다.

## 무엇을 저장하나

레코드별로 **연속 신호 전체**에 구획기를 돌리고(비트 조각이 아니라), 각 비트의
P 위치를 **비트 좌표(0~299 · R = 100)** 로 되돌려 저장한다.

| 필드 | 뜻 |
|---|---|
| `p_idx` | 비트 안 P 위치(0~299). P 창 밖이거나 못 찾으면 **−1** |
| `p_score` | 국소 두드러짐(Q7-U 의 `score_at` 과 같은 자) |
| `r_samp` | 재현한 R 표본 위치(360Hz 연속 신호 좌표) — 재현 검증용 |
| `pid`·`sym` | 정합 증명에 쓴 신원 — 소비 측에서 다시 확인할 수 있게 |

⚠️ **이 런은 어떤 가설도 검정하지 않는다.** 관문이 없다. 자산을 만들고
**정합을 증명**할 뿐이다. 판정은 Q7-S′ 가 한다.

⚠️ **유도 0 만 구획한다**(Q7-U 와 동일). 2유도를 다 하면 70분이라 1회 캐시로도
Colab 세션이 위험하다. 캐시 형식은 유도 1 을 나중에 덧붙일 수 있게 열어 둔다.

★ **레코드마다 중간 저장**한다 — 35분짜리 셀이 Colab 에서 끊겨도 이어서 돈다.

In [ ]:
# CELL 0 — 공용
import numpy as np

class AssetError(RuntimeError): pass

def show(a, k=8):
    a = np.asarray(a)
    return f"{a[:k].tolist()}{' ...' if len(a) > k else ''}"

print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록 (관문 없음 — 자산 생성 런)
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches()
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ── ★ `mit-bih/svdb_labels.py` 의 비트 절단 상수를 **그대로** 옮긴다.
#    값이 어긋나면 재현이 깨지고, 그건 아래 정합 검사가 잡는다.
FS_SRC, FS_DST = 128, 360
L, RPRE = 300, 100
RALIGN_MS = 50
AAMI5 = {'N': 0, 'L': 0, 'R': 0, 'e': 0, 'j': 0,
         'A': 1, 'a': 1, 'J': 1, 'S': 1,
         'V': 2, 'E': 2,
         'F': 3,
         '/': 4, 'f': 4, 'Q': 4}
BEAT_SYMS = set("NLRBAaJSVrFejnE/fQ?")

# ── P 구획 (Q7-U 에서 이긴 설정: **원신호** · NeuroKit2 dwt · 유도 0)
LEAD = 0
P_LO_MS, P_HI_MS = -278.0, -42.0        # Q7-U 와 같은 P 창(R 기준)
DELIN_METHOD = "dwt"

SV5   = os.path.join(MITBIH, "svdb_data5.npz")
DLDIR = "/content/svdb_raw"
OUTNP = os.path.join(MITBIH, "svdb_pdelin.npz")
CKPT  = os.path.join(MITBIH, "svdb_pdelin_ckpt.npz")

CONFIG = dict(
    exp="quest46_q7p0_svdb_pdelin", quest="ailab-2026-0046", step="svdb-pdelin",
    parent_exp=["quest46_q7u_public_delineator"],
    purpose=("Q7-S′ 의 **전제 조건**. Q7-U 가 세운 자(NeuroKit2 웨이블릿 구획 · BUT PDB "
             "전문가 주석 대비 Se 0.9109)는 **연속 신호**를 먹는데 `svdb_data5.npz` 는 "
             "**300샘플 비트 조각**만 갖고 있다. Q7-U 가 조각 되붙이기를 실측으로 "
             "기각했으므로(F1 0.7705 → 0.6132) **원 SVDB 연속 신호**에서 구획한다. "
             "그 결과를 비트 좌표로 되돌리려면 비트 순서가 맞아야 하는데 npz 에 R 위치가 "
             "없다 — 그래서 `svdb_labels.py` 의 절단 로직을 재현하고 **(pid, sym) 수열 "
             "일치**로 정합을 **구성적으로 증명**한다"),
    dataset="MIT-BIH SVDB 78레코드(원 128Hz → 360Hz) + svdb_data5.npz(비트 신원)",
    fs_src=FS_SRC, fs_dst=FS_DST, beat_len=L, rpre=RPRE, ralign_ms=RALIGN_MS,
    lead=LEAD, p_win_ms=[P_LO_MS, P_HI_MS], delineator=DELIN_METHOD,
    ruler=dict(exp="quest46_q7u_public_delineator", arm="dwt|raw",
               se=0.9109, ppv=0.7145, f1=0.7705, db="BUT PDB"),
    gates="**없음** — 자산 생성 런이다. 가설은 Q7-S′ 가 검정한다",
    rule_check={
        "R16 fallback 없음": "SVDB·svdb_data5.npz 없으면 **중단** — 합성으로 대체 안 함",
        "R22 누수 없음":     "구획기는 **라벨을 인자로도 안 받는다**. R 위치만 쓴다",
        "R27 ③ 좌표 정합":   "★★ **(pid, sym) 수열이 npz 와 원소 단위로 같아야** 한다. "
                             "어긋나면 즉시 중단 — 이 퀘스트는 좌표로 두 번 데었다",
        "R29 ② 분기 금지":   "관문이 없으므로 결론 분기도 없다",
    },
    caveat=("★ **유도 0 만** 구획한다(Q7-U 와 동일) — 2유도면 70분이라 Colab 세션이 "
            "위험하다. 캐시 형식은 유도 1 을 나중에 덧붙일 수 있게 열어 둔다. "
            "★ **소거는 안 쓴다** — Q7-U 의 U2 가 소거를 검출 경로에서 내렸다"
            "(격자 70칸 전부 음수). 소거 잔차는 Q7-S′ 에서 **별도 특징**으로만 들어간다. "
            "★ 레코드마다 **중간 저장**한다 — 끊겨도 이어서 돈다. "
            "★ 학습 0회 · 약 35분(실측: 30분 레코드 1개당 구획 17~24초 x 78)"))
run = MedKOSRun("quest46_q7p0_svdb_pdelin", CONFIG, project=PROJECT)
run.log("설정 ✅ **자산 생성 런** — 관문 없음. 이 런의 절반은 **좌표 정합의 증명**이다")
run.log(f"  자 = Q7-U 의 `dwt|raw`(BUT PDB Se 0.9109 / PPV 0.7145) · 유도 {LEAD}")
run.log(f"  좌표: 원 {FS_SRC}Hz → {FS_DST}Hz · 비트 {L}샘플 · R = idx {RPRE}")
run.log(f"  P 창 {P_LO_MS:.0f}~{P_HI_MS:.0f}ms (R 기준) — Q7-U 와 동일")
for k_, v_ in CONFIG["rule_check"].items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 【P0-A】 `svdb_data5.npz` 적재 — **필드를 추측하지 않고 확인한다**
run.log("\n" + "=" * 100)
run.log("【P0-A】 비트 신원 적재 (svdb_data5.npz)")
run.log("=" * 100)
if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음 — `mit-bih/svdb_labels.py` 의 build 를 먼저 돌린다(R16)")
D5 = np.load(SV5, allow_pickle=True)
run.log(f"  보유 필드: {sorted(D5.files)}")           # ★ 추측 금지 — 있는 걸 본다
for need in ("beat", "pid", "sym", "y3"):
    if need not in D5.files:
        raise AssetError(f"`{need}` 가 없다 — 정합 증명을 못 한다. 보유: {sorted(D5.files)}")
PID5 = np.asarray(D5["pid"]).astype(int)
SYM5 = np.asarray(D5["sym"]).astype(str)
Y3   = np.asarray(D5["y3"]).astype(int)
NB5  = len(PID5)
if np.asarray(D5["beat"]).shape[1:] != (2, L):
    raise AssetError(f"비트 모양이 (n,2,{L}) 가 아니다 — {np.asarray(D5['beat']).shape}")
run.log(f"  비트 {NB5:,} · 레코드(pid) {len(np.unique(PID5))} · 기호 {sorted(set(SYM5.tolist()))}")
run.log(f"  pid {show(PID5)}")
run.log(f"  sym {show(SYM5)}")
run.log(f"  y3 분포 {dict(zip(*[x.tolist() for x in np.unique(Y3, return_counts=True)]))}")
CONFIG["d5"] = dict(n_beat=int(NB5), n_pid=int(len(np.unique(PID5))),
                    fields=sorted(D5.files))
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【P0-B】 ★★ 비트 절단 재현 + **좌표 정합의 구성적 증명**
try:
    import wfdb
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)
    importlib.invalidate_caches(); import wfdb
from scipy.signal import resample_poly

run.log("\n" + "=" * 100)
run.log("【P0-B】 비트 절단 재현 — `svdb_labels.py` 와 같은 순서로")
run.log("=" * 100)
os.makedirs(DLDIR, exist_ok=True)
RECS = [str(r) for r in wfdb.get_record_list("svdb")]          # ⛔ 실패 시 중단(R16)
if len(RECS) < 50:
    raise AssetError(f"SVDB 목록이 {len(RECS)}개 — 다운로드 실패")
run.log(f"  레코드 {len(RECS)}개")

def realign(sig, s, T):
    """`svdb_labels.py` 와 **같은** R 재정합 — ±50ms 창 벡터크기 argmax(라벨 미사용)."""
    w = int(FS_DST * RALIGN_MS / 1000.0)
    a, b = max(0, s - w), min(T, s + w + 1)
    if b - a < 3:
        return s
    return a + int(np.argmax(np.sqrt(sig[0, a:b] ** 2 + sig[1, a:b] ** 2)))

def load_rec(rec):
    """원 SVDB 을 받아 360Hz 연속 신호와 **유효 비트의 R 위치**를 돌려준다."""
    for ext in ("hea", "dat", "atr"):
        fp = os.path.join(DLDIR, f"{rec}.{ext}")
        if not os.path.exists(fp):
            wfdb.dl_files("svdb", DLDIR, [f"{rec}.{ext}"])
    r = wfdb.rdrecord(os.path.join(DLDIR, rec))
    ann = wfdb.rdann(os.path.join(DLDIR, rec), "atr")
    if r.p_signal is None or r.p_signal.shape[1] < 2:
        raise AssetError(f"{rec}: 2유도가 아니다")
    fs_src = int(getattr(r, "fs", FS_SRC))
    sig = np.nan_to_num(np.asarray(r.p_signal, float)[:, :2].T, nan=0.0)
    if fs_src != FS_DST:
        sig = np.stack([resample_poly(sig[c], FS_DST, fs_src) for c in range(2)])
    T = sig.shape[1]
    samp = (np.asarray(ann.sample) * (FS_DST / fs_src)).astype(int)
    sym = np.asarray(ann.symbol)
    bm = np.array([s in BEAT_SYMS for s in sym])
    bs = np.array([realign(sig, int(s), T) for s in samp[bm]]) if bm.any() else np.zeros(0, int)
    bsym = sym[bm]
    keep = np.array([AAMI5.get(bsym[i]) is not None
                     and bs[i] - RPRE >= 0 and bs[i] - RPRE + L <= T
                     for i in range(len(bs))], bool)
    return sig, bs, bsym, keep

T0 = time.time()
R_ALL, PID_ALL, SYM_ALL, SIGS = [], [], [], {}
for ri, rec in enumerate(RECS):
    try:
        sig, bs, bsym, keep = load_rec(rec)
    except Exception as e:
        run.log(f"  ✗ {rec}: {type(e).__name__} {e}")       # 재현 실패도 그대로 남긴다
        continue
    SIGS[ri] = (rec, sig, bs, bsym, keep)
    R_ALL.append(bs[keep]); PID_ALL.append(np.full(int(keep.sum()), ri))
    SYM_ALL.append(bsym[keep])
    if (ri + 1) % 20 == 0 or ri == len(RECS) - 1:
        run.log(f"  {ri+1}/{len(RECS)}  누적 비트 {sum(len(x) for x in R_ALL):,} "
                f"({time.time()-T0:.0f}초)")
R_ALL = np.concatenate(R_ALL); PID_ALL = np.concatenate(PID_ALL)
SYM_ALL = np.concatenate(SYM_ALL).astype(str)
run.log(f"  재현 비트 {len(R_ALL):,} · {time.time()-T0:.0f}초")

# ── ★★★ 정합 증명 — 어긋나면 **즉시 중단**한다(R27 ③)
run.log("\n  ★★ 좌표 정합 증명 — `(pid, sym)` 수열이 npz 와 원소 단위로 같은가")
if len(R_ALL) != NB5:
    raise AssetError(f"비트 수 불일치 — 재현 {len(R_ALL):,} vs npz {NB5:,}. "
                     "절단 로직이 `svdb_labels.py` 와 다르다")
bad_p = int((PID_ALL != PID5).sum()); bad_s = int((SYM_ALL != SYM5).sum())
if bad_p or bad_s:
    i0 = int(np.argmax((PID_ALL != PID5) | (SYM_ALL != SYM5)))
    raise AssetError(f"수열 불일치 — pid {bad_p}개 · sym {bad_s}개. 첫 어긋남 idx {i0}: "
                     f"재현 (pid {PID_ALL[i0]}, sym {SYM_ALL[i0]}) vs "
                     f"npz (pid {PID5[i0]}, sym {SYM5[i0]})")
run.log(f"    ✅ **{NB5:,} 비트 전부 일치** — pid·sym 수열이 원소 단위로 같다.")
run.log("       → 「npz 의 비트 i」 ↔ 「내가 되살린 R 위치 i」 가 **증명됐다**")
CONFIG["align"] = dict(n=int(NB5), pid_mismatch=0, sym_mismatch=0, proven=True)
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — 【P0-C】 P 구획 — **연속 신호**에 Q7-U 의 자를 건다 (약 35분 · 중간 저장)
try:
    import neurokit2 as nk
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neurokit2"], check=True)
    importlib.invalidate_caches(); import neurokit2 as nk

run.log("\n" + "=" * 100)
run.log(f"【P0-C】 P 구획 — NeuroKit2 `{DELIN_METHOD}` · **원 연속 신호** · 유도 {LEAD}")
run.log("=" * 100)
run.log("  ▸ 비트 조각을 되붙인 신호가 아니다 — Q7-U 가 그걸 기각했다(F1 0.7705 → 0.6132)")
run.log("  ▸ 구획기는 **라벨을 인자로도 안 받는다**. R 위치만 준다(R22)")

def ms2s(ms):
    return int(round(abs(ms) * FS_DST / 1000.0)) * (1 if ms >= 0 else -1)

def _detrend(v):
    n = len(v)
    if n < 3:
        return v - np.mean(v) if n else v
    A = np.c_[np.arange(n, dtype=float), np.ones(n)]
    return v - A @ np.linalg.lstsq(A, v, rcond=None)[0]

SCORE_HALF_MS = 100.0        # 점수 창 반폭 — P 폭(80~110ms)보다 크고 QRS 는 안 닿는다

def score_at(x, pos):
    """검출 위치의 **국소 두드러짐** = |추세제거 진폭| / 창 안 MAD.

    ★ Q7-U 의 동명 함수와 **창 정의가 다르다**. Q7-U 는 창을 `[q+P_LO, q+P_HI]`(둘 다
      음수 = 위치 **앞쪽**)로 잡고 인덱스를 클램프해서, 실제로는 **창 끝**의 값을 쟀다.
      Q7-U 안에서는 모든 팔이 같은 자를 써서 비교가 유효했지만(문턱 결정용 상대값),
      여기 `p_score` 는 Q7-S′ 가 **특징으로 쓸** 값이라 의미가 맞아야 한다.
      그래서 창을 **위치 중심 ±SCORE_HALF_MS** 로 바꾼다(픽스처 ⑬ 이 잡았다)."""
    w = ms2s(SCORE_HALF_MS)
    out = []
    for q in pos:
        a, b = max(int(q) - w, 0), min(int(q) + w + 1, len(x))
        if b - a < 3:
            out.append(0.0); continue
        seg = _detrend(x[a:b])
        m = float(np.median(np.abs(seg - np.median(seg)))) + 1e-12
        out.append(float(abs(seg[int(q) - a]) / m))
    return np.asarray(out, float)

CK = {}
if os.path.exists(CKPT):                              # ★ 중간 저장에서 이어서
    z = np.load(CKPT, allow_pickle=True)
    CK = {int(k): z[k] for k in z.files}
    run.log(f"  ▸ 중간 저장에서 이어감 — 완료 레코드 {len(CK)}개")

T1_ = time.time()
for ri, (rec, sig, bs, _bsym, _keep) in sorted(SIGS.items()):
    if ri in CK:
        continue
    x = sig[LEAD]
    try:
        _, w = nk.ecg_delineate(x, rpeaks=bs, sampling_rate=FS_DST, method=DELIN_METHOD)
        pk = np.asarray([p if (p is not None and np.isfinite(p)) else -1
                         for p in w.get("ECG_P_Peaks", [])], float)
    except Exception as e:
        run.log(f"  ✗ {rec}: {type(e).__name__} {e}")
        pk = np.full(len(bs), -1.0)
    # 구획기 출력 길이가 R 개수와 다를 수 있다 — 위치로 되맞춘다(추측 금지)
    cand = pk[(pk >= 0)].astype(int)
    lo, hi = ms2s(P_LO_MS), ms2s(P_HI_MS)
    p_abs = np.full(len(bs), -1, int)
    for i, R in enumerate(bs):
        m = cand[(cand >= R + lo) & (cand < R + hi)]
        if len(m):
            p_abs[i] = int(m[np.argmin(np.abs(m - (R + lo + hi) // 2))])
    sc = np.where(p_abs >= 0, score_at(x, np.clip(p_abs, 0, len(x) - 1)), 0.0)
    # ★ 라벨에서 나온 `keep`(AAMI 유효성)은 **여기 안 넣는다** — 이 루프는 R 위치만 본다(R22).
    #   유효성 마스크는 아래 복원 단계에서 SIGS 로부터 붙인다.
    CK[ri] = np.stack([bs.astype(float), p_abs.astype(float), sc])
    np.savez(CKPT, **{str(k): v for k, v in CK.items()})     # ★ 레코드마다 저장
    if (len(CK) % 10 == 0) or ri == max(SIGS):
        run.log(f"  {len(CK)}/{len(SIGS)}  ({time.time()-T1_:.0f}초)")
run.log(f"  구획 완료 · {time.time()-T1_:.0f}초")

# ── 비트 좌표로 되돌린다 — 정합이 증명됐으므로 순서는 npz 와 같다
P_IDX, P_SC, R_SMP = [], [], []
for ri, (rec, sig, bs, bsym, keep) in sorted(SIGS.items()):
    a = CK[ri]
    r_, pa_, sc_ = a[0].astype(int), a[1].astype(int), a[2]
    kp_ = keep                                          # ★ 유효성은 여기서 붙인다(구획과 분리)
    if len(kp_) != len(r_):
        raise AssetError(f"{rec}: 마스크 길이 {len(kp_)} != 비트 {len(r_)}")
    rel = np.where(pa_ >= 0, pa_ - (r_ - RPRE), -1)
    rel = np.where((rel >= 0) & (rel < L), rel, -1)          # 비트 창 밖은 −1
    P_IDX.append(rel[kp_]); P_SC.append(sc_[kp_]); R_SMP.append(r_[kp_])
P_IDX = np.concatenate(P_IDX).astype(np.int32)
P_SC = np.concatenate(P_SC).astype("float32")
R_SMP = np.concatenate(R_SMP).astype(np.int64)
if len(P_IDX) != NB5:
    raise AssetError(f"되돌린 비트 수 {len(P_IDX):,} != npz {NB5:,}")
run.log(f"  비트 좌표 복원 {len(P_IDX):,} · P 발견 {int((P_IDX >= 0).sum()):,} "
        f"({(P_IDX >= 0).mean():.3f})")

In [ ]:
# CELL 5 — 【P0-D】 진단 · 저장 · 마무리
run.log("\n" + "=" * 100)
run.log("【P0-D】 진단 — 이 표가 말이 되는가")
run.log("=" * 100)
ok = P_IDX >= 0
run.log(f"  P 발견률 {ok.mean():.4f} (BUT PDB 문헌 P/QRS ≈ 0.71 · Q7-U 발화율 1.0 기준)")
pos_ms = (P_IDX[ok] - RPRE) / FS_DST * 1000.0
run.log(f"  P 위치 (R 기준) 중앙 {np.median(pos_ms):+.0f}ms · "
        f"IQR {np.percentile(pos_ms, 25):+.0f} ~ {np.percentile(pos_ms, 75):+.0f}ms")
run.log("  ▸ 생리학적으로 PR 은 120~200ms — 중앙값이 그 근처가 아니면 좌표가 틀린 것이다")
for c, nm in ((0, "N"), (1, "S"), (2, "V")):
    m = (Y3 == c)
    if m.sum() < 50:
        continue
    mm = m & ok
    run.log(f"    {nm}  n={int(m.sum()):>7,} · P 발견률 {mm.sum()/max(m.sum(),1):.4f} · "
            f"P 위치 중앙 {np.median((P_IDX[mm]-RPRE)/FS_DST*1000):+.0f}ms · "
            f"점수 중앙 {np.median(P_SC[mm]):.2f}")
run.log("  ▸ **여기서 결론을 내지 않는다** — S/N 차이가 보여도 판정은 Q7-S′ 가 한다(관문 없음)")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.4))
ax[0].hist(pos_ms, bins=60, color="tab:blue")
ax[0].axvline(-120, ls=":", color="k"); ax[0].axvline(-200, ls=":", color="k")
ax[0].set_xlabel("P peak, ms relative to R  (dotted = physiologic PR 120-200ms)")
ax[0].set_ylabel("beats"); ax[0].grid(alpha=.3)
lab = [("N", 0), ("S", 1), ("V", 2)]
ax[1].bar(range(len(lab)), [float((ok & (Y3 == c)).sum()) / max(int((Y3 == c).sum()), 1)
                            for _, c in lab])
ax[1].set_xticks(range(len(lab))); ax[1].set_xticklabels([n for n, _ in lab])
ax[1].set_ylabel("P found rate"); ax[1].set_ylim(0, 1); ax[1].grid(alpha=.3, axis="y")
rr = np.asarray(D5["pre_rr"], float)
ax[2].scatter(rr[ok][::37], pos_ms[::37], s=2, alpha=.25)
ax[2].set_xlabel("pre-RR (samples @360Hz)"); ax[2].set_ylabel("P peak, ms rel. R")
ax[2].grid(alpha=.3)
fig.tight_layout()
PNG = run.save_fig("q7p0_svdb_pdelin", fig)
plt.close(fig); display(Image(PNG))

np.savez(OUTNP, p_idx=P_IDX, p_score=P_SC, r_samp=R_SMP,
         pid=PID5, sym=SYM5, lead=np.array([LEAD]),
         p_win_ms=np.array([P_LO_MS, P_HI_MS]), fs=np.array([FS_DST]),
         beat_len=np.array([L]), rpre=np.array([RPRE]),
         source=np.array(["quest46_q7p0_svdb_pdelin"]))
run.log(f"\n  ✅ 저장 {OUTNP}  ({len(P_IDX):,} 비트)")
run.log("     소비 측(Q7-S′)은 `pid`·`sym` 을 svdb_data5.npz 와 다시 대조해 정합을 재확인할 것")

CONFIG["out"] = dict(path=OUTNP, n=int(len(P_IDX)), found=float(ok.mean()),
                     pos_med_ms=float(np.median(pos_ms)))
run.save_json("config", CONFIG)
run.finish({
    "exp_id": "quest46_q7p0_svdb_pdelin",
    "metric": "svdb_p_found_rate", "value": float(ok.mean()), "passed": True,
    "summary": ("Q7-S′ 의 전제 조건 — SVDB 전 비트의 P 위치 표. Q7-U 가 세운 자를 "
                "**원 연속 신호**에 걸고, `(pid, sym)` 수열 일치로 좌표 정합을 "
                "**구성적으로 증명**한 뒤 비트 좌표로 되돌렸다. **관문 없음**."),
    "align": CONFIG.get("align", {}), "d5": CONFIG.get("d5", {}),
    "out": CONFIG.get("out", {}), "ruler": CONFIG.get("ruler", {}),
    "rule_check": CONFIG.get("rule_check", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step svdb-pdelin` → 그리고 **Q7-S′**")